Sequential Chats and Customer Onboarding

## Setup

In [ ]:
from utils import get_openai_api_key
OPENAI_API_KEY = get_openai_api_key()

In [ ]:
llm_config = {
    "config_list": [
        {
            "model": "gemini-2.0-flash-lite",
            "api_key": "", 
            "api_type": "google",
            # Forces the framework to wait out the 429 restriction
            "max_retries": 5,          
            "timeout": 60
        }
    ]
}


In [42]:
from autogen import ConversableAgent

## Creating the needed agents

In [43]:

onboarding_personal_information_agent = ConversableAgent(
    name="onboarding_personal_information_agent",  # Removed spaces
    system_message='''You are a helpful customer onboarding agent.
    You are here to help new customers get started with our product.
    Your job is to gather customer's name and location.
    Do not ask for other information. Return 'TERMINATE' 
    when you have gathered all the information.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
)

onboarding_topic_preference_agent = ConversableAgent(
    name="onboarding_topic_preference_agent",  # Removed spaces
    system_message='''You are a helpful customer onboarding agent.
    You are here to help new customers get started with our product.
    Your job is to gather customer's preferences on news topics.
    Do not ask for other information.
    Return 'TERMINATE' when you have gathered all the information.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
)

customer_engagement_agent = ConversableAgent(
    name="customer_engagement_agent",  # Removed spaces
    system_message='''You are a helpful customer service agent.
    You are here to provide fun for the customer based on the user's
    personal information and topic preferences.
    This could include fun facts, jokes, or interesting stories.
    Make sure to make it engaging and fun!
    Return 'TERMINATE' when you are done.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
    is_termination_msg=lambda msg: "terminate" in msg.get("content").lower(),
)

customer_proxy_agent = ConversableAgent(
    name="customer_proxy_agent",
    llm_config=False,
    code_execution_config=False,
    human_input_mode="ALWAYS",
    is_termination_msg=lambda msg: "terminate" in msg.get("content").lower(),
)


## Creating tasks

Now, you can craft a series of tasks to facilitate the onboarding process.

In [44]:
chats = [
    {
        "sender": onboarding_personal_information_agent,
        "recipient": customer_proxy_agent,
        "message": "Hello, I'm here to help you get started with our product. Could you tell me your name and location?",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Return the customer information as JSON object only: {'name': '', 'location': ''}",
        },
        "max_turns": 1,
        "clear_history": True
    },
    {
        "sender": onboarding_topic_preference_agent,
        "recipient": customer_proxy_agent,
        "message": "Great! Could you tell me what topics you are interested in reading about?",
        "summary_method": "reflection_with_llm",
        "max_turns": 1,
        "clear_history": False
    },
    {
        "sender": customer_proxy_agent,
        "recipient": customer_engagement_agent,
        "message": "Let's find something fun to read.",
        "max_turns": 1,
        "summary_method": "reflection_with_llm",
    },
]

## Start the onboarding process

**Note**: You might get a slightly different response than what's shown in the video. Feel free to try different inputs, such as name, location, and preferences.

In [45]:
from autogen import initiate_chats

chat_results = initiate_chats(chats)



********************************************************************************
Starting a new chat....

********************************************************************************
onboarding_personal_information_agent (to customer_proxy_agent):

Hello, I'm here to help you get started with our product. Could you tell me your name and location?

--------------------------------------------------------------------------------
customer_proxy_agent (to onboarding_personal_information_agent):

My name is Paul Immanuel and I am from India

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (efea4677-f91e-4867-94ce-7b6d8c439cbd): Maximum turns (1) reached

********************************************************************************
Starting a new chat....

********************************************************************************
onboarding_topic_preference_agent (to customer_proxy_agent):

Great! Could you tell me what

c:\Users\Paul\anaconda3\envs\cuda_gpu\Lib\site-packages\autogen\agentchat\conversable_agent.py:2216: UserWarning: Cannot extract summary using reflection_with_llm: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\nPlease retry in 39.41367105s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.co

customer_proxy_agent (to onboarding_topic_preference_agent):

Interested in Tech stack news

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (1aa4064e-e7ce-4897-9dfd-3b99aa16a85d): Maximum turns (1) reached

********************************************************************************
Starting a new chat....

********************************************************************************
customer_proxy_agent (to customer_engagement_agent):

Let's find something fun to read.
Context: 



--------------------------------------------------------------------------------


c:\Users\Paul\anaconda3\envs\cuda_gpu\Lib\site-packages\autogen\agentchat\conversable_agent.py:2216: UserWarning: Cannot extract summary using reflection_with_llm: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\nPlease retry in 35.310441829s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.c

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite\nPlease retry in 35.03223517s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@t

## Print out the summary

In [ ]:
summary=[]
for chat_result in chat_results:
    print(chat_result.summary)
    summary.append(chat_result.summary)
    print("\n")

{'name': 'Ramisha', 'location': 'Theni'}


Ramisha from Theni is interested in reading about the Law of the Universe.


According to the Law of Attraction, positive or negative thoughts bring positive or negative experiences into a person's life.




In [ ]:
summary[0]

"{'name': 'Ramisha', 'location': 'Theni'}"

In [ ]:
#!pip install pandas

In [ ]:
import pandas as pd
import ast

# Input string
#data_str = "{'name': 'Ramisha', 'location': 'Coimbatore'}"
data_str=summary[0]
# Convert string to dictionary
data_dict = ast.literal_eval(data_str)

# Convert dictionary to DataFrame
df = pd.DataFrame([data_dict])

df


,name,location
0,Ramisha,Theni


In [ ]:
df["TopicPreference"]=summary[1]

In [ ]:
df

,name,location,TopicPreference
0,Ramisha,Theni,Ramisha from Theni is interested in reading ab...


In [ ]:
df["Enagement"]=summary[2]

In [ ]:
df

,name,location,TopicPreference,Enagement
0,Ramisha,Theni,Ramisha from Theni is interested in reading ab...,"According to the Law of Attraction, positive o..."


In [ ]:
#!pip install gspread
#!pip install oauth2client

In [ ]:
def add_leads_to_google_sheet(df):
    import gspread
    from oauth2client.service_account import ServiceAccountCredentials

    # Define the scope for Google Sheets API
    scope = ['https://www.googleapis.com/auth/spreadsheets']

    # Add credentials to the account
    creds = ServiceAccountCredentials.from_json_keyfile_name(
        'websiteleadchatgptapi-a46f50ae1507.json', scope
    )

    # Authorize the client
    client = gspread.authorize(creds)

    # Specify the Google Sheet ID
    sheet_id = '1w-19wp0W2nRV5Pde5gim6kKlCoPTjv6mT3A8lAdNrcM'

    # Open the spreadsheet using its ID
    spreadsheet = client.open_by_key(sheet_id)

    # Select the first worksheet (index 0)
    worksheet = spreadsheet.get_worksheet(0)

    # Append each row of the DataFrame to the worksheet
    for _, row in df.iterrows():
        # Convert row to a list and append to the sheet
        worksheet.append_row(row.tolist())

    print("Data added successfully!")
    return "added"


In [ ]:
add_leads_to_google_sheet(df)

Data added successfully!


'added'

## Print out the cost

In [ ]:
for chat_result in chat_results:
    print(chat_result.cost)
    print("\n")

{'usage_including_cached_inference': {'total_cost': 0.000115, 'gpt-3.5-turbo-0125': {'cost': 0.000115, 'prompt_tokens': 140, 'completion_tokens': 30, 'total_tokens': 170}}, 'usage_excluding_cached_inference': {'total_cost': 5.75e-05, 'gpt-3.5-turbo-0125': {'cost': 5.75e-05, 'prompt_tokens': 70, 'completion_tokens': 15, 'total_tokens': 85}}}


{'usage_including_cached_inference': {'total_cost': 0.000152, 'gpt-3.5-turbo-0125': {'cost': 0.000152, 'prompt_tokens': 202, 'completion_tokens': 34, 'total_tokens': 236}}, 'usage_excluding_cached_inference': {'total_cost': 0.000152, 'gpt-3.5-turbo-0125': {'cost': 0.000152, 'prompt_tokens': 202, 'completion_tokens': 34, 'total_tokens': 236}}}


{'usage_including_cached_inference': {'total_cost': 0.0005819999999999999, 'gpt-3.5-turbo-0125': {'cost': 0.0005819999999999999, 'prompt_tokens': 558, 'completion_tokens': 202, 'total_tokens': 760}}, 'usage_excluding_cached_inference': {'total_cost': 0.00029099999999999997, 'gpt-3.5-turbo-0125': {'cost': 0.